In [ ]:
"""Per-item embeddings for Cuisync soft matching (v2).

Run in Colab (paste into a cell, or upload and `%run vectorize_softmatch.py`) or locally.
Writes the SAME file names as before, so app.py / softmatch.py need no loader changes:
    <feature>_item_vecs.npy, <feature>_items.json, embeddings_order.json

What's new (each is a switch in CONFIG so you can A/B them with evaluate_softmatch.py):
  NORMALIZE        strip quantities / prep words / synonyms so 'chopped fresh ginger' == 'ginger'
  CONTEXT_PREFIX   embed 'food ingredient: pepper' instead of 'pepper' (disambiguates short words)
  CENTER_VECTORS   remove the shared 'it is food' direction so unrelated items stop looking
                   40-50% similar (this is what FLOOR was compensating for)
  SOFT_DF          document frequency counted over *similar* items, so 'chicken thigh' isn't
                   treated as rare just because the exact string is rare
Change ONE switch at a time, rebuild, and compare scores on your gold pairs.
"""
import json
import os
import re
from collections import Counter

import numpy as np

CONFIG = {
    "MODEL_NAME": "all-MiniLM-L6-v2",   # also try "all-mpnet-base-v2" or "BAAI/bge-base-en-v1.5"
    "NORMALIZE": True,
    "CONTEXT_PREFIX": True,
    "CENTER_VECTORS": True,
    "SOFT_DF": True,
    "SOFT_DF_THRESHOLD": 0.60,           # items at/above this similarity share document frequency
}

FOLDER = "/content/drive/MyDrive/CuisyncCleaning/"   # change if running locally

FIELDS = {                      # feature name -> recipe field
    "ingredients": "ingredient_text",
    "actions":     "action_text",
    "cookware":    "cookware_text",
    "utensils":    "utensil_text",
}

CONTEXT = {
    "ingredients": "food ingredient",
    "actions": "cooking technique",
    "cookware": "cookware",
    "utensils": "kitchen utensil",
}

# Map regional / variant names onto one canonical name. Fill this from the "most frequent items"
# printout below; a few examples are included. Keys must already be normalised (lower case).
SYNONYMS = {
    "shoyu": "soy sauce", "toyo": "soy sauce", "ganjang": "soy sauce",
    "scallion": "spring onion", "green onion": "spring onion",
    "coriander leaves": "cilantro", "capsicum": "bell pepper",
    "prawn": "shrimp", "prawns": "shrimp",
}

# Words that describe preparation, not identity. Deliberately conservative: 'ground', 'dried',
# 'smoked', 'sweet' etc. are NOT removed because they change the ingredient.
_PREP = {"chopped", "minced", "sliced", "diced", "grated", "crushed", "finely", "roughly",
         "coarsely", "freshly", "fresh", "large", "small", "medium", "optional", "peeled",
         "cubed", "shredded", "thinly"}
_COUNTERS = {"clove", "cloves", "pinch", "dash", "bunch", "sprig", "sprigs", "stalk", "stalks",
             "slice", "slices", "piece", "pieces", "can", "cans"}   # only dropped after a number
_QTY = re.compile(r"\b\d+(?:[./]\d+)?\s*(?:g|kg|mg|ml|l|oz|lb|lbs|tsp|tbsp|cups?)?\b")


def normalize_item(text, synonyms=SYNONYMS):
    """'  2 tbsp Chopped FRESH Ginger (optional) ' -> 'ginger'."""
    s = re.sub(r"\([^)]*\)", " ", text.lower())
    had_qty = bool(_QTY.search(s))
    s = _QTY.sub(" ", s)
    s = re.sub(r"[^a-z' -]", " ", s)
    words = [w for w in s.split() if w not in _PREP]
    if had_qty and len(words) > 1 and words[0] in _COUNTERS:     # '2 cloves garlic' -> 'garlic'
        words = words[1:]
        if words[0] == "of" and len(words) > 1:
            words = words[1:]
    s = " ".join(words) if words else text.strip().lower()
    return synonyms.get(s, s)


def split_items(text, normalize=True):
    """'Chicken Breast, Ginger ,chicken breast' -> ['chicken breast', 'ginger'] (order kept)."""
    seen, items = set(), []
    for x in (text or "").split(","):
        x = normalize_item(x) if normalize else x.strip().lower()
        if x and x not in seen:
            seen.add(x)
            items.append(x)
    return items


def compute_mean(vecs, df):
    """Frequency-weighted mean embedding (weighted by how many recipes use each item)."""
    w = np.asarray(df, dtype=np.float64)
    w = w / w.sum()
    return (vecs * w[:, None]).sum(axis=0)


def apply_center(vecs, mean):
    v = vecs - mean
    return (v / np.maximum(np.linalg.norm(v, axis=1, keepdims=True), 1e-8)).astype("float32")


def soft_document_frequency(vecs, recipe_item_ids, threshold, chunk=1024):
    """For each item: how many recipes contain that item OR a similar one (cosine >= threshold)."""
    from scipy import sparse
    n_items, n_recipes = len(vecs), len(recipe_item_ids)
    rows = np.concatenate([np.asarray(ids, dtype=np.int64) for ids in recipe_item_ids if len(ids)])
    cols = np.concatenate([np.full(len(ids), r) for r, ids in enumerate(recipe_item_ids) if len(ids)])
    member = sparse.csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)),
                               shape=(n_items, n_recipes))
    out = np.zeros(n_items)
    for s in range(0, n_items, chunk):
        near = sparse.csr_matrix((vecs[s:s + chunk] @ vecs.T >= threshold).astype(np.float32))
        out[s:s + chunk] = (near @ member).getnnz(axis=1)   # recipes reachable via a near item
    return out


def main():
    from sentence_transformers import SentenceTransformer
    try:                                   # in Colab: connect to Google Drive
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:                    # running locally: nothing to mount
        pass

    with open(os.path.join(FOLDER, "recipes.json"), "r", encoding="utf-8") as f:
        recipes = json.load(f)
    model = SentenceTransformer(CONFIG["MODEL_NAME"])

    def embed(texts, name):
        if CONFIG["CONTEXT_PREFIX"]:
            texts = [f"{CONTEXT[name]}: {t}" for t in texts]
        return model.encode(texts, batch_size=64, normalize_embeddings=True,
                            show_progress_bar=True).astype("float32")

    means = {}
    for name, field in FIELDS.items():
        per_recipe = [split_items(r.get(field, ""), CONFIG["NORMALIZE"]) for r in recipes]
        vocab = sorted({item for items in per_recipe for item in items})
        index = {item: i for i, item in enumerate(vocab)}
        recipe_ids = [[index[i] for i in items] for items in per_recipe]
        doc_freq = Counter(item for items in per_recipe for item in items)
        df_exact = np.array([doc_freq[item] for item in vocab], dtype=np.float64)

        vecs = embed(vocab, name)
        if CONFIG["CENTER_VECTORS"]:
            means[name] = compute_mean(vecs, df_exact)
            vecs = apply_center(vecs, means[name])
        df = (soft_document_frequency(vecs, recipe_ids, CONFIG["SOFT_DF_THRESHOLD"])
              if CONFIG["SOFT_DF"] else df_exact)

        np.save(os.path.join(FOLDER, f"{name}_item_vecs.npy"), vecs)
        with open(os.path.join(FOLDER, f"{name}_items.json"), "w", encoding="utf-8") as f:
            json.dump({"items": vocab, "df": [float(x) for x in df],
                       "df_exact": [int(x) for x in df_exact],
                       "n_recipes": len(recipes), "recipes": recipe_ids,
                       # the app recounts "df" the same way when the admin adds dishes
                       "soft_df_threshold": CONFIG["SOFT_DF_THRESHOLD"] if CONFIG["SOFT_DF"] else None,
                       "config": CONFIG},
                      f, ensure_ascii=False)
        print(f"{name}: {len(vocab)} distinct items, vectors {vecs.shape}")
        if name == "ingredients":
            print("  most frequent (check these for synonyms to add to SYNONYMS):",
                  [w for w, _ in doc_freq.most_common(25)])

    with open(os.path.join(FOLDER, "embeddings_order.json"), "w", encoding="utf-8") as f:
        json.dump([[r.get("title", ""), r.get("country", "")] for r in recipes], f, ensure_ascii=False)

    # ---- Probe: use this to choose FLOOR. Shows raw vs the vectors actually saved. ----
    PAIRS = [("chicken", "chicken breast"), ("chicken breast", "chicken thigh"),
             ("ginger", "ginger root"), ("chicken", "pork"), ("chicken", "fish"),
             ("chicken", "flour"), ("soy sauce", "fish sauce"), ("garlic", "sugar"),
             ("kecap manis", "dark soy sauce"), ("lemongrass", "lime leaves")]
    pre = CONTEXT["ingredients"] + ": " if CONFIG["CONTEXT_PREFIX"] else ""
    print("\nPair similarity (cosine, as saved):")
    for a, b in PAIRS:
        va, vb = model.encode([pre + normalize_item(a), pre + normalize_item(b)], normalize_embeddings=True)
        if CONFIG["CENTER_VECTORS"]:
            va, vb = apply_center(np.stack([va, vb]), means["ingredients"])
        print(f"  {a:16s} vs {b:16s} {float(va @ vb):.2f}")
    print("\nDone. Copy ALL *_item_vecs.npy, *_items.json and embeddings_order.json next to app.py.")
    print("If CENTER_VECTORS is on, lower FLOOR in softmatch.py (try 0.1) and tune it on gold pairs.")


if __name__ == "__main__":
    main()